# ⭐ Day 90: Transformers & Attention Mechanisms - Building a Text Classifier
### Day 90 of 369-day Python & AI Learning Path

📝 Welcome to Day 90 of your incredible 369-day journey! Today, we explore one of the most important innovations in modern AI — the **Transformer architecture** and the **Attention Mechanism**. Since their introduction in the seminal paper "Attention Is All You Need" (2017), Transformers have revolutionized Natural Language Processing, computer vision, and beyond. We will build a Transformer-based text classifier from scratch, visualize attention weights, and then leverage pre-trained models from Hugging Face to achieve state-of-the-art results. Let's unlock the power of attention! 🚀

## 📚 Table of Contents
1. [Introduction to Transformers and Self-Attention](#1)
2. [Understanding the Attention Mechanism](#2)
3. [Loading and Preprocessing Text Dataset](#3)
4. [Implementing Multi-Head Self-Attention from Scratch](#4)
5. [Building a Transformer-based Text Classifier](#5)
6. [Training the Model with PyTorch](#6)
7. [Evaluation & Performance Analysis](#7)
8. [Using Pre-trained Hugging Face Transformers](#8)
9. [Fine-tuning a Pre-trained Transformer](#9)
10. [Comparison: Custom vs Pre-trained](#10)
11. [Hands-On Exercises](#11)
12. [Solutions](#12)
13. [Summary & Day 91 Teaser](#13)

<a id='1'></a>
## 🧠 1. Introduction to Transformers and Self-Attention

### Why Transformers Changed Everything
Before Transformers, sequence modeling relied on **RNNs** and **LSTMs**, which process data sequentially. This created two major problems:
1. **Slow training**: Cannot parallelize across time steps
2. **Long-range dependencies**: Information from distant tokens gets diluted

### The Transformer Revolution
The Transformer architecture, introduced by Vaswani et al. in 2017, solves both problems by:
- Using **Self-Attention** to capture relationships between ALL tokens simultaneously
- Enabling **massive parallelization** during training
- Achieving **state-of-the-art results** across virtually every NLP task

### Key Components
| Component | Purpose |
|-----------|---------|
| **Self-Attention** | Computes weighted relationships between all token pairs |
| **Multi-Head Attention** | Runs attention multiple times in parallel for richer representations |
| **Positional Encoding** | Injects sequence order information (since attention itself is order-agnostic) |
| **Feed-Forward Networks** | Applies non-linear transformations to each position independently |
| **Layer Normalization** | Stabilizes training by normalizing layer inputs |

💡 **Fun Fact**: GPT, BERT, T5, LLaMA, Claude, and virtually every modern LLM are built on the Transformer architecture!

<a id='2'></a>
## 🔍 2. Understanding the Attention Mechanism

### The Core Idea
Attention allows the model to "focus" on relevant parts of the input when producing each output. In **Self-Attention**, each token attends to every other token in the sequence.

### Scaled Dot-Product Attention
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

Where:
- **Q (Query)**: What am I looking for?
- **K (Key)**: What do I contain?
- **V (Value)**: What information do I provide?
- $d_k$: Dimension of the key vectors (scaling factor prevents softmax saturation)

### Intuition
Imagine reading a sentence: "The cat sat on the mat because it was tired." When processing the word "it", attention allows the model to strongly connect "it" with "cat" (not "mat"), resolving the pronoun reference.

### Multi-Head Attention
Instead of one attention operation, we run **h parallel attention heads**, each learning different types of relationships:
- Head 1 might learn syntactic relationships
- Head 2 might learn semantic relationships
- Head 3 might learn long-range dependencies

$$ \text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O $$

🚀 This multi-perspective view is what makes Transformers so expressive!

<a id='3'></a>
## 📦 3. Loading and Preprocessing Text Dataset

We will use the **AG News** dataset — a collection of 120,000 news articles across 4 categories: World, Sports, Business, and Sci/Tech. It's perfect for text classification and more interesting than binary sentiment analysis!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import time
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"✅ PyTorch version: {torch.__version__}")
print(f"🚀 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# Download AG News dataset
!pip install -q torchtext
from torchtext.datasets import AG_NEWS

train_iter, test_iter = AG_NEWS(split=('train', 'test'))

# Convert iterators to lists for easier handling
train_data = list(train_iter)
test_data = list(test_iter)

print(f"📊 Training samples: {len(train_data)}")
print(f"📊 Test samples: {len(test_data)}")

# Label mapping
label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
print(f"📋 Classes: {list(label_names.values())}")

# Show sample
sample_label, sample_text = train_data[0]
print(f"\n📝 Sample text: {sample_text[:150]}...")
print(f"🏷️  Label: {label_names[sample_label]}")

# Distribution of classes
train_labels = [label for label, _ in train_data]
label_counts = Counter(train_labels)

plt.figure(figsize=(8, 5))
plt.bar([label_names[k] for k in sorted(label_names.keys())], 
        [label_counts[k] for k in sorted(label_names.keys())],
        color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
plt.title('📊 Class Distribution in AG News Training Set', fontsize=14, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(axis='y', alpha=0.3)
for i, k in enumerate(sorted(label_names.keys())):
    plt.text(i, label_counts[k] + 500, str(label_counts[k]), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Text Preprocessing and Vocabulary Building
# ============================================================
def tokenize(text):
    """Simple tokenizer: lowercase, remove special chars, split by spaces."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

# Build vocabulary from training data
counter = Counter()
for _, text in train_data:
    counter.update(tokenize(text))

# Keep top 20,000 words
vocab_size = 20000
most_common = counter.most_common(vocab_size - 2)  # Reserve 2 for special tokens
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

vocab_size = len(vocab)
print(f"📚 Vocabulary size: {vocab_size:,}")
print(f"   Most common words: {list(vocab.keys())[:10]}")

# Text-to-tensor conversion
def text_to_tensor(text, max_len=256):
    tokens = tokenize(text)
    indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    # Pad or truncate
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return torch.tensor(indices, dtype=torch.long)

# Create PyTorch Dataset
class AGNewsDataset(Dataset):
    def __init__(self, data, max_len=256):
        self.data = data
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        label, text = self.data[idx]
        # Convert labels from 1-4 to 0-3
        label_tensor = torch.tensor(label - 1, dtype=torch.long)
        text_tensor = text_to_tensor(text, self.max_len)
        return text_tensor, label_tensor

# Create datasets and dataloaders
max_seq_len = 256
batch_size = 64

train_dataset = AGNewsDataset(train_data, max_len=max_seq_len)
test_dataset = AGNewsDataset(test_data, max_len=max_seq_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"✅ DataLoaders created!")
print(f"   Batches per epoch (train): {len(train_loader)}")
print(f"   Batches per epoch (test): {len(test_loader)}")

# Verify a batch
sample_batch_text, sample_batch_labels = next(iter(train_loader))
print(f"\n📦 Sample batch shapes:")
print(f"   Text tensor: {sample_batch_text.shape}")
print(f"   Labels: {sample_batch_labels.shape}")
print(f"   Labels in batch: {sample_batch_labels[:5]}")

<a id='4'></a>
## 🏗️ 4. Implementing Multi-Head Self-Attention from Scratch

This is the mathematical heart of the Transformer! We will implement every component manually to build deep intuition.

In [ ]:
# ============================================================
# Positional Encoding
# ============================================================
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding as described in the original Transformer paper.
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# ============================================================
# Multi-Head Self-Attention
# ============================================================
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Self-Attention mechanism.
    Splits d_model into num_heads heads, each with d_k = d_model / num_heads.
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = np.sqrt(self.d_k)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V: (batch, num_heads, seq_len, d_k)
        Returns: attention output and attention weights
        """
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (batch, heads, seq, seq)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        output = torch.matmul(attn_weights, V)  # (batch, heads, seq, d_k)
        return output, attn_weights
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        seq_len = query.size(1)
        
        # Linear projections and reshape for multi-head
        Q = self.W_q(query).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model)
        
        # Final linear projection
        output = self.W_o(attn_output)
        return output, attn_weights

print("✅ PositionalEncoding and MultiHeadAttention modules defined!")
print("💡 These are the core building blocks of the Transformer architecture.")

In [ ]:
# ============================================================
# Transformer Encoder Block
# ============================================================
class TransformerEncoderBlock(nn.Module):
    """
    A single Transformer encoder block:
    Multi-Head Attention → Add & Norm → Feed Forward → Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Feed-forward network
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_out, attn_weights = self.attention(x, x, x, mask)
        x = self.norm1(x + attn_out)
        
        # Feed-forward with residual connection
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        
        return x, attn_weights

# ============================================================
# Complete Transformer Classifier
# ============================================================
class TransformerClassifier(nn.Module):
    """
    Transformer-based text classifier for AG News.
    """
    def __init__(self, vocab_size, d_model=128, num_heads=4, 
                 num_layers=2, d_ff=512, num_classes=4, 
                 max_len=256, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Transformer encoder blocks
        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x, return_attention=False):
        # Create padding mask
        mask = (x != 0).unsqueeze(1).unsqueeze(1)  # (batch, 1, 1, seq_len)
        
        # Embedding + positional encoding
        x = self.embedding(x) * np.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Pass through encoder blocks
        attention_weights = []
        for block in self.encoder_blocks:
            x, attn = block(x, mask)
            attention_weights.append(attn)
        
        # Global average pooling over sequence dimension
        # Mask out padding tokens
        mask_expanded = mask.squeeze(1).squeeze(1).unsqueeze(-1).float()  # (batch, seq_len, 1)
        x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        
        # Classification
        logits = self.classifier(x)
        
        if return_attention:
            return logits, attention_weights
        return logits

# Instantiate model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerClassifier(
    vocab_size=vocab_size,
    d_model=128,
    num_heads=4,
    num_layers=2,
    d_ff=512,
    num_classes=4,
    max_len=max_seq_len,
    dropout=0.1
).to(device)

print(f"✅ TransformerClassifier created!")
print(f"   Device: {device}")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Test forward pass
test_input = torch.randint(0, vocab_size, (2, max_seq_len)).to(device)
test_output = model(test_input)
print(f"\n🧪 Test forward pass:")
print(f"   Input shape: {test_input.shape}")
print(f"   Output shape: {test_output.shape}")
print(f"   Output (sample): {test_output[0]}")

<a id='5'></a>
## 🏗️ 5. Building a Transformer-based Text Classifier

Our model is assembled! It consists of:
1. **Token Embedding**: Converts word indices to dense vectors
2. **Positional Encoding**: Adds position information
3. **2 Transformer Encoder Blocks**: Each with Multi-Head Attention + Feed-Forward
4. **Global Average Pooling**: Aggregates sequence information
5. **Classification Head**: Maps to 4 news categories

Now let's set up training!

<a id='6'></a>
## ⚔️ 6. Training the Model with PyTorch

We'll train our custom Transformer with cross-entropy loss, Adam optimizer, and learning rate scheduling.

In [ ]:
# ============================================================
# Training Setup
# ============================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

num_epochs = 10

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (texts, labels) in enumerate(dataloader):
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if (batch_idx + 1) % 200 == 0:
            print(f"   Batch {batch_idx+1}/{len(dataloader)} | Loss: {loss.item():.4f} | Acc: {100.*correct/total:.2f}%")
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for texts, labels in dataloader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy, all_preds, all_labels

print("🚀 Starting Training...")
print(f"   Epochs: {num_epochs}")
print(f"   Batch Size: {batch_size}")
print(f"   Learning Rate: {optimizer.param_groups[0]['lr']}")
print("-" * 60)

train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(1, num_epochs + 1):
    start_time = time.time()
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    epoch_time = time.time() - start_time
    print(f"\n📊 Epoch {epoch}/{num_epochs} | Time: {epoch_time:.1f}s")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print("-" * 60)

print("\n🎉 Training Complete!")

<a id='7'></a>
## 📊 7. Evaluation & Performance Analysis

Let's visualize training curves, attention heatmaps, confusion matrix, and sample predictions!

In [ ]:
# ============================================================
# Plot 1: Training Curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', color='#FF6B6B', linewidth=2, marker='o')
axes[0].plot(val_losses, label='Val Loss', color='#4ECDC4', linewidth=2, marker='s')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('📈 Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(train_accs, label='Train Accuracy', color='#FF6B6B', linewidth=2, marker='o')
axes[1].plot(val_accs, label='Val Accuracy', color='#4ECDC4', linewidth=2, marker='s')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('📈 Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# Plot 2: Confusion Matrix
# ============================================================
_, _, all_preds, all_labels = evaluate(model, test_loader, criterion, device)

cm = confusion_matrix(all_labels, all_preds)
class_names_list = ['World', 'Sports', 'Business', 'Sci/Tech']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names_list, yticklabels=class_names_list,
            cbar_kws={'label': 'Count'})
plt.title('🔍 Confusion Matrix - Custom Transformer', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

# Classification report
print("\n📋 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names_list, digits=4))

final_acc = accuracy_score(all_labels, all_preds)
print(f"\n✅ Final Test Accuracy: {final_acc * 100:.2f}%")

In [ ]:
# ============================================================
# Plot 3: Attention Weight Heatmaps
# ============================================================
def visualize_attention(model, text, vocab_inv, max_len=256):
    """Visualize attention weights for a given text."""
    model.eval()
    
    # Convert text to tensor
    tokens = tokenize(text)
    indices = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    
    input_tensor = torch.tensor([indices], dtype=torch.long).to(device)
    
    with torch.no_grad():
        _, attention_weights = model(input_tensor, return_attention=True)
    
    # Get non-padded tokens
    valid_len = len(tokens)
    display_tokens = tokens[:min(valid_len, 30)]  # Show first 30 tokens
    
    # Average attention across all heads for the last encoder layer
    attn = attention_weights[-1][0].mean(dim=0).cpu().numpy()  # (seq_len, seq_len)
    attn = attn[:len(display_tokens), :len(display_tokens)]
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(attn, xticklabels=display_tokens, yticklabels=display_tokens,
                cmap='viridis', cbar_kws={'label': 'Attention Weight'})
    plt.title('🧠 Attention Heatmap (Last Layer, Averaged Heads)', fontsize=14, fontweight='bold')
    plt.xlabel('Key Tokens', fontsize=12)
    plt.ylabel('Query Tokens', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()
    
    return attention_weights

# Visualize attention on a sample text
sample_text = "The stock market rallied today as tech companies reported strong quarterly earnings."
print(f"📝 Analyzing attention for: \"{sample_text}\"")
attn_weights = visualize_attention(model, sample_text, None)
print("💡 Brighter colors indicate stronger attention relationships between tokens.")

In [ ]:
# ============================================================
# Plot 4: Sample Predictions
# ============================================================
def show_predictions(model, dataset, num_samples=8):
    """Display sample texts with predicted and true labels."""
    model.eval()
    
    # Get random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx, dataset_idx in enumerate(indices):
        text_tensor, true_label = dataset[dataset_idx]
        text_tensor = text_tensor.unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(text_tensor)
            probs = F.softmax(output, dim=1)
            pred_label = output.argmax(dim=1).item()
            confidence = probs[0][pred_label].item()
        
        # Get original text
        _, original_text = test_data[dataset_idx]
        
        # Determine color
        color = '#2ECC71' if pred_label == true_label else '#E74C3C'
        
        axes[idx].text(0.5, 0.7, original_text[:200] + '...', 
                        ha='center', va='center', fontsize=9, wrap=True,
                        transform=axes[idx].transAxes)
        axes[idx].text(0.5, 0.3, f"True: {class_names_list[true_label]}", 
                        ha='center', va='center', fontsize=11, fontweight='bold',
                        transform=axes[idx].transAxes, color='#2C3E50')
        axes[idx].text(0.5, 0.15, f"Pred: {class_names_list[pred_label]} ({confidence*100:.1f}%)", 
                        ha='center', va='center', fontsize=11, fontweight='bold',
                        transform=axes[idx].transAxes, color=color)
        axes[idx].set_xlim(0, 1)
        axes[idx].set_ylim(0, 1)
        axes[idx].axis('off')
        
        # Add border color
        for spine in axes[idx].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
    
    plt.suptitle('📝 Sample Predictions: Custom Transformer', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(model, test_dataset, num_samples=8)
print("✅ Green borders = correct predictions, Red borders = incorrect predictions.")

<a id='8'></a>
## 🤗 8. Using Pre-trained Hugging Face Transformers

Now let's see how pre-trained models compare! We'll use **DistilBERT** — a smaller, faster version of BERT that retains 97% of its performance while being 60% faster.

In [ ]:
# ============================================================
# Install and Import Hugging Face Libraries
# ============================================================
!pip install -q transformers datasets

from transformers import (
    DistilBertTokenizer, DistilBertForSequenceClassification,
    BertTokenizer, BertForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

print("✅ Hugging Face libraries installed and imported!")

# Load AG News dataset in Hugging Face format
hf_dataset = load_dataset("ag_news")
print(f"📊 HF Dataset loaded:")
print(f"   Train: {len(hf_dataset['train'])} samples")
print(f"   Test: {len(hf_dataset['test'])} samples")

# Initialize DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
print(f"\n🤖 DistilBERT tokenizer loaded!")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print("✅ Dataset tokenized and formatted for PyTorch!")

# Show sample tokenization
sample = tokenized_datasets['train'][0]
print(f"\n📦 Sample tokenized input:")
print(f"   Input IDs shape: {sample['input_ids'].shape}")
print(f"   Attention mask shape: {sample['attention_mask'].shape}")
print(f"   Label: {sample['labels']}")

<a id='9'></a>
## 🔧 9. Fine-tuning a Pre-trained Transformer

We'll fine-tune DistilBERT on AG News. Notice how we only need a few epochs because the model already understands language from pre-training on massive corpora!

In [ ]:
# ============================================================
# Load Pre-trained DistilBERT with Classification Head
# ============================================================
pretrained_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=4,
    id2label={0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'},
    label2id={'World': 0, 'Sports': 1, 'Business': 2, 'Sci/Tech': 3}
).to(device)

print(f"✅ DistilBERT model loaded!")
print(f"   Total parameters: {sum(p.numel() for p in pretrained_model.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad):,}")

# ============================================================
# Training Arguments
# ============================================================
training_args = TrainingArguments(
    output_dir="./distilbert_agnews",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",  # Disable wandb/tensorboard
    seed=42
)

# Metric computation
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create Trainer
trainer = Trainer(
    model=pretrained_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(10000)),  # Subset for speed
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42).select(range(2000)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Starting DistilBERT fine-tuning...")
print("   (Using 10k train / 2k eval samples for demonstration speed)")
print("-" * 60)

# Train!
trainer.train()

print("\n🎉 Fine-tuning Complete!")

In [ ]:
# ============================================================
# Evaluate Fine-tuned DistilBERT
# ============================================================
eval_results = trainer.evaluate()
print("📊 DistilBERT Evaluation Results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

# Get predictions for confusion matrix
predictions_output = trainer.predict(tokenized_datasets["test"].select(range(2000)))
preds = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

# Confusion Matrix
cm_pretrained = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_pretrained, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names_list, yticklabels=class_names_list,
            cbar_kws={'label': 'Count'})
plt.title('🔍 Confusion Matrix - Fine-tuned DistilBERT', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📋 Classification Report (DistilBERT):")
print(classification_report(labels, preds, target_names=class_names_list, digits=4))

pretrained_acc = accuracy_score(labels, preds)
print(f"\n✅ DistilBERT Test Accuracy: {pretrained_acc * 100:.2f}%")

<a id='10'></a>
## ⚖️ 10. Comparison: Custom Transformer vs Pre-trained Models

Let's compare our from-scratch Transformer against the fine-tuned DistilBERT across multiple dimensions.

In [ ]:
# ============================================================
# Comprehensive Comparison
# ============================================================
comparison_data = {
    'Metric': [
        'Test Accuracy (%)',
        'Training Time (epochs)',
        'Model Parameters',
        'Pre-training Data',
        'Requires Tokenizer',
        'Implementation Complexity',
        'Interpretability',
        'Deployment Speed'
    ],
    'Custom Transformer': [
        f"{final_acc * 100:.2f}",
        "10 epochs",
        f"{sum(p.numel() for p in model.parameters()):,}",
        "None (trained from scratch)",
        "Custom (simple)",
        "High (manual implementation)",
        "High (full control)",
        "Fast (small model)"
    ],
    'DistilBERT (Fine-tuned)': [
        f"{pretrained_acc * 100:.2f}",
        "3 epochs",
        f"{sum(p.numel() for p in pretrained_model.parameters()):,}",
        "Wikipedia + BookCorpus",
        "DistilBERT Tokenizer",
        "Low (library abstraction)",
        "Medium (black-box tendencies)",
        "Slower (larger model)"
    ]
}

# Display as table
import pandas as pd
df_comparison = pd.DataFrame(comparison_data)
print("📊 Model Comparison:")
print(df_comparison.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
models = ['Custom\nTransformer', 'Fine-tuned\nDistilBERT']
accuracies = [final_acc * 100, pretrained_acc * 100]
colors = ['#FF6B6B', '#4ECDC4']

bars = axes[0].bar(models, accuracies, color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('🎯 Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

# Parameter count comparison (log scale)
params = [sum(p.numel() for p in model.parameters()), 
          sum(p.numel() for p in pretrained_model.parameters())]
bars2 = axes[1].bar(models, params, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Parameters (millions)', fontsize=12)
axes[1].set_title('📦 Model Size Comparison', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(axis='y', alpha=0.3)
for bar, param in zip(bars2, params):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
                f'{param/1e6:.1f}M', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print("\n💡 Key Insight: Pre-trained models leverage massive pre-training to achieve superior performance")
print("   with less task-specific data, while custom models offer full transparency and control.")

<a id='11'></a>
## 🛠️ Hands-On Exercises

Now it's your turn to deepen your understanding! Complete these 4 challenges:

### Exercise 1: 🧮 Attention Pattern Analysis
Modify the attention visualization to display individual attention heads (not averaged). Compare how different heads attend to different syntactic or semantic relationships in the same sentence.

### Exercise 2: ⚡ Positional Encoding Experiment
Replace the sinusoidal positional encoding with **learnable positional embeddings** (nn.Embedding). Train the model for 5 epochs and compare convergence speed and final accuracy against the sinusoidal version.

### Exercise 3: 🏗️ Deeper Architecture
Increase the number of Transformer encoder layers from 2 to 4, and increase d_model from 128 to 256. Observe how this affects training time, overfitting tendencies, and final accuracy. Document your findings.

### Exercise 4: 🤗 BERT vs DistilBERT
Fine-tune the full **BERT-base-uncased** model (not DistilBERT) on the same AG News subset. Compare its accuracy, training time, and parameter count against DistilBERT. Is the performance gain worth the extra computational cost?

<a id='12'></a>
## ✅ Solutions

Below are complete, working solutions for all four exercises. Study them carefully!

In [ ]:
# ============================================================
# ✅ SOLUTION 1: Individual Attention Head Visualization
# ============================================================
def visualize_attention_heads(model, text, max_len=256):
    """Visualize each attention head separately."""
    model.eval()
    tokens = tokenize(text)
    indices = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    
    input_tensor = torch.tensor([indices], dtype=torch.long).to(device)
    
    with torch.no_grad():
        _, attention_weights = model(input_tensor, return_attention=True)
    
    # Last layer attention: (batch, heads, seq, seq)
    last_layer_attn = attention_weights[-1][0].cpu().numpy()
    valid_len = min(len(tokens), 20)  # Show first 20 tokens
    display_tokens = tokens[:valid_len]
    
    num_heads = last_layer_attn.shape[0]
    fig, axes = plt.subplots(1, num_heads, figsize=(20, 4))
    
    for h in range(num_heads):
        attn = last_layer_attn[h, :valid_len, :valid_len]
        sns.heatmap(attn, xticklabels=display_tokens, yticklabels=display_tokens,
                    cmap='plasma', ax=axes[h], cbar=False)
        axes[h].set_title(f'Head {h+1}', fontsize=10, fontweight='bold')
        axes[h].tick_params(axis='x', rotation=45, labelsize=7)
        axes[h].tick_params(axis='y', labelsize=7)
    
    plt.suptitle('🧠 Individual Attention Heads (Last Layer)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

sample_text = "The athlete scored a goal in the final minute of the championship game."
print(f"📝 Analyzing individual heads for: \"{sample_text}\"")
visualize_attention_heads(model, sample_text)
print("✅ Solution 1 complete! Notice how different heads focus on different relationships.")

In [ ]:
# ============================================================
# ✅ SOLUTION 2: Learnable Positional Embeddings
# ============================================================
class LearnablePositionalEncoding(nn.Module):
    """Learnable positional embeddings instead of sinusoidal."""
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.pos_embedding = nn.Embedding(max_len, d_model)
    
    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = x + self.pos_embedding(positions)
        return self.dropout(x)

# Rebuild model with learnable positional encoding
class TransformerClassifierLearnablePos(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, 
                 num_layers=2, d_ff=512, num_classes=4, 
                 max_len=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = LearnablePositionalEncoding(d_model, max_len, dropout)
        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x, return_attention=False):
        mask = (x != 0).unsqueeze(1).unsqueeze(1)
        x = self.embedding(x) * np.sqrt(self.d_model)
        x = self.pos_encoding(x)
        attention_weights = []
        for block in self.encoder_blocks:
            x, attn = block(x, mask)
            attention_weights.append(attn)
        mask_expanded = mask.squeeze(1).squeeze(1).unsqueeze(-1).float()
        x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        logits = self.classifier(x)
        if return_attention:
            return logits, attention_weights
        return logits

# Train for 5 epochs
model_learnable = TransformerClassifierLearnablePos(
    vocab_size=vocab_size, d_model=128, num_heads=4, 
    num_layers=2, d_ff=512, num_classes=4, max_len=max_seq_len
).to(device)

optimizer_lp = optim.Adam(model_learnable.parameters(), lr=1e-3, weight_decay=1e-5)
criterion_lp = nn.CrossEntropyLoss()

print("🚀 Training with Learnable Positional Embeddings (5 epochs)...")
lp_losses, lp_accs = [], []

for epoch in range(1, 6):
    model_learnable.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer_lp.zero_grad()
        outputs = model_learnable(texts)
        loss = criterion_lp(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_learnable.parameters(), max_norm=1.0)
        optimizer_lp.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    acc = 100. * correct / total
    lp_losses.append(avg_loss)
    lp_accs.append(acc)
    print(f"   Epoch {epoch} | Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

# Compare
print("\n📊 Comparison: Sinusoidal vs Learnable Positional Encoding")
print(f"   Sinusoidal (Epoch 5): Loss={train_losses[4]:.4f}, Acc={train_accs[4]:.2f}%")
print(f"   Learnable (Epoch 5):  Loss={lp_losses[-1]:.4f}, Acc={lp_accs[-1]:.2f}%")
print("✅ Solution 2 complete! Learnable embeddings often converge faster but may overfit on small data.")

In [ ]:
# ============================================================
# ✅ SOLUTION 3: Deeper Architecture (4 layers, d_model=256)
# ============================================================
model_deep = TransformerClassifier(
    vocab_size=vocab_size,
    d_model=256,
    num_heads=8,
    num_layers=4,
    d_ff=1024,
    num_classes=4,
    max_len=max_seq_len,
    dropout=0.2  # Higher dropout to combat overfitting
).to(device)

print(f"✅ Deep Transformer created!")
print(f"   Parameters: {sum(p.numel() for p in model_deep.parameters()):,}")
print(f"   (vs {sum(p.numel() for p in model.parameters()):,} for original)")

optimizer_deep = optim.Adam(model_deep.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler_deep = optim.lr_scheduler.StepLR(optimizer_deep, step_size=3, gamma=0.5)
criterion_deep = nn.CrossEntropyLoss()

print("\n🚀 Training Deep Transformer (5 epochs)...")
deep_train_losses, deep_val_losses = [], []
deep_train_accs, deep_val_accs = [], []

for epoch in range(1, 6):
    model_deep.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer_deep.zero_grad()
        outputs = model_deep(texts)
        loss = criterion_deep(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_deep.parameters(), max_norm=1.0)
        optimizer_deep.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_loss = total_loss / len(train_loader)
    train_acc = 100. * correct / total
    
    # Validation
    model_deep.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for texts, labels in test_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model_deep(texts)
            loss = criterion_deep(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_loss = val_loss / len(test_loader)
    val_acc = 100. * val_correct / val_total
    scheduler_deep.step()
    
    deep_train_losses.append(train_loss)
    deep_val_losses.append(val_loss)
    deep_train_accs.append(train_acc)
    deep_val_accs.append(val_acc)
    
    print(f"   Epoch {epoch} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

# Plot comparison
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses[:5], label='Original (2L, 128D)', marker='o')
plt.plot(deep_train_losses, label='Deep (4L, 256D)', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(val_accs[:5], label='Original (2L, 128D)', marker='o')
plt.plot(deep_val_accs, label='Deep (4L, 256D)', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Val Accuracy (%)')
plt.title('Validation Accuracy Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Solution 3 complete! Deeper models have more capacity but require careful regularization.")

In [ ]:
# ============================================================
# ✅ SOLUTION 4: BERT-base vs DistilBERT Comparison
# ============================================================
from transformers import BertForSequenceClassification, BertTokenizer

# Load BERT tokenizer and model
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=4,
    id2label={0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'},
    label2id={'World': 0, 'Sports': 1, 'Business': 2, 'Sci/Tech': 3}
).to(device)
 print(f"✅ BERT-base loaded!")
print(f"   Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")
print(f"   (vs {sum(p.numel() for p in pretrained_model.parameters()):,} for DistilBERT)")

# Tokenize with BERT tokenizer
def tokenize_bert(examples):
    return bert_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized_bert = hf_dataset.map(tokenize_bert, batched=True)
tokenized_bert = tokenized_bert.remove_columns(["text"])
tokenized_bert = tokenized_bert.rename_column("label", "labels")
tokenized_bert.set_format("torch")

# Train BERT (1 epoch for speed demonstration)
 bert_training_args = TrainingArguments(
    output_dir="./bert_agnews",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # Smaller batch for BERT's size
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs_bert",
    logging_steps=100,
    report_to="none",
    seed=42
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,
    train_dataset=tokenized_bert["train"].shuffle(seed=42).select(range(5000)),
    eval_dataset=tokenized_bert["test"].shuffle(seed=42).select(range(1000)),
    tokenizer=bert_tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=bert_tokenizer),
    compute_metrics=compute_metrics,
)

print("\n🚀 Fine-tuning BERT-base (1 epoch, 5k samples)...")
 bert_trainer.train()

# Evaluate
 bert_eval = bert_trainer.evaluate()
 bert_preds_output = bert_trainer.predict(tokenized_bert["test"].select(range(1000)))
 bert_preds = np.argmax(bert_preds_output.predictions, axis=-1)
 bert_labels = bert_preds_output.label_ids
 bert_acc = accuracy_score(bert_labels, bert_preds)

print("\n📊 Final Comparison:")
print(f"   DistilBERT: {pretrained_acc*100:.2f}% accuracy | {sum(p.numel() for p in pretrained_model.parameters())/1e6:.1f}M params")
print(f"   BERT-base:  {bert_acc*100:.2f}% accuracy | {sum(p.numel() for p in bert_model.parameters())/1e6:.1f}M params")
print(f"   Custom Transformer: {final_acc*100:.2f}% accuracy | {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

print("\n✅ Solution 4 complete! BERT is larger and slower but often more accurate. DistilBERT offers the best efficiency trade-off.")

<a id='13'></a>
## 🌟 Summary & Day 91 Teaser

### What You Accomplished Today 🎉
Congratulations! On Day 90, you have:
- ✅ Understood the revolutionary Transformer architecture and Self-Attention mechanism
- ✅ Implemented Multi-Head Attention, Positional Encoding, and Transformer Encoder blocks from scratch
- ✅ Built and trained a complete Transformer-based text classifier on AG News
- ✅ Visualized attention heatmaps to interpret model behavior
- ✅ Fine-tuned DistilBERT from Hugging Face and compared it against your custom model
- ✅ Analyzed the trade-offs between custom implementations and pre-trained models
- ✅ Completed 4 hands-on exercises with full solutions

### Key Takeaways 💡
1. **Attention is all you need**: The ability to relate any token to any other token in parallel is transformative.
2. **Pre-training matters**: Models like BERT leverage massive corpora to understand language before task-specific fine-tuning.
3. **Interpretability**: Attention weights provide a window into *what* the model is focusing on.
4. **Engineering trade-offs**: Accuracy, speed, size, and interpretability must be balanced for each use case.

### 🚀 Teaser for Day 91
Tomorrow, on **Day 91**, we will dive into **Large Language Models (LLMs) and Prompt Engineering**! You'll learn how to interact with models like GPT-4, craft effective prompts, build few-shot learning pipelines, and understand the emerging discipline of prompt engineering. It's the practical bridge between research and real-world AI applications—see you there!

---
⭐ **You are 90 days into an extraordinary 369-day journey. Every day you are building the skills to shape the future of AI. Keep going!** ⭐